# 07 — Industrialización del pipeline de abastecimiento

**Tostao | Forecasting SKU–Tienda + incertidumbre + política Q90**

## Objetivo
Convertir los notebooks experimentales en un flujo modular, reproducible y ejecutable. El código de negocio vive en `src/tostao_supply_chain/`; este notebook sirve como punto de entrada, configuración, ejecución y revisión de resultados.

### Flujo
1. Validar archivos, columnas, tipos y llaves.
2. Construir panel diario SKU–Tienda.
3. Entrenar ETS y pronosticar los próximos siete días.
4. Estimar incertidumbre con errores ETS fuera de muestra.
5. Calcular Q85/Q90/Q95 con límite de cobertura de ocho días.
6. Exportar pronósticos, recomendaciones y resumen de ejecución.

**Importante:** el pipeline requiere `backtesting_modelos_base.csv`, generado por el Notebook 03. Los resultados son recomendaciones de apoyo a la decisión, no órdenes automáticas de compra.

## 1. Preparar entorno y archivos

En Google Colab, sube el ZIP del proyecto y descomprímelo. Después coloca en la carpeta `data/` los siguientes archivos:
- `ventas_historicas.csv`
- `inventario_actual.csv`
- `catalogo_productos.csv`
- `maestro_tiendas.csv`
- `backtesting_modelos_base.csv` (exportado por el Notebook 03)

En ejecución local, clona o descomprime el repositorio y coloca esos mismos archivos en `data/`.

In [ ]:
from pathlib import Path
import sys
import zipfile
import shutil
import pandas as pd
import numpy as np

# Ajusta PROJECT_DIR si descomprimiste el proyecto en otra ubicación.
candidates = [
    Path.cwd() / "tostao_supply_chain_project",
    Path.cwd(),
    Path("/content/drive/MyDrive/EMPLEO/TOSTAO/tostao_supply_chain_project"),
    Path("/mnt/data/tostao_supply_chain_project")
]
PROJECT_DIR = next((p for p in candidates if (p / "src" / "tostao_supply_chain").exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("No se encontró la carpeta del proyecto. Descomprime el ZIP y ajusta PROJECT_DIR.")

sys.path.insert(0, str(PROJECT_DIR / "src"))
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

print("Proyecto:", PROJECT_DIR)
print("Datos:", DATA_DIR)
print("Salidas:", OUTPUT_DIR)

Proyecto: /content/drive/MyDrive/EMPLEO/TOSTAO/tostao_supply_chain_project
Datos: /content/drive/MyDrive/EMPLEO/TOSTAO/tostao_supply_chain_project/data
Salidas: /content/drive/MyDrive/EMPLEO/TOSTAO/tostao_supply_chain_project/outputs


## 2. Instalar dependencias e importar el pipeline

En Colab o en un entorno nuevo, instala las dependencias declaradas en `requirements.txt`. En local también puedes instalar el proyecto con `pip install -e .` para habilitar el comando CLI.

In [ ]:
# En Google Colab, descomenta la siguiente línea si aún no instalaste dependencias:
# %pip install -r requirements.txt

from tostao_supply_chain.config import PipelineConfig
from tostao_supply_chain.pipeline import run_pipeline
from tostao_supply_chain.data_validation import read_inputs, validate_inputs

print("Paquete importado correctamente.")

Paquete importado correctamente.


## 3. Validación previa de entradas

Antes de ejecutar el modelo, inspeccionamos dimensiones y llaves. La validación detiene el proceso si hay columnas faltantes, llaves duplicadas en inventario/maestros, valores negativos o combinaciones SKU–Tienda desalineadas.

In [ ]:
input_data = validate_inputs(read_inputs(DATA_DIR))
for name, frame in input_data.items():
    print(f"{name:>10}: {frame.shape}")

     sales: (14560, 4)
 inventory: (160, 3)
  products: (8, 6)
    stores: (20, 3)
  backtest: (13440, 7)


## 4. Configuración de negocio

Parámetros iniciales:
- Horizonte: 7 días.
- Protección adicional: 1 día.
- Política base: Q90.
- Comparativos: Q85 y Q95.
- Simulaciones: 10.000.
- Pool de errores por producto: mínimo 30; si no, pool global.
- Semilla: 42 para reproducibilidad.

La cobertura de ocho días es una hipótesis inicial que debe validarse con capacidad, lead time, frecuencia de despacho y perecibilidad.

In [ ]:
CONFIG = PipelineConfig(
    horizon_days=7,
    safety_days=1,
    service_levels=(0.85, 0.90, 0.95),
    default_service_level=0.90,
    n_simulations=10_000,
    min_product_errors=30,
    random_seed=42
)
CONFIG

PipelineConfig(horizon_days=7, safety_days=1, service_levels=(0.85, 0.9, 0.95), default_service_level=0.9, n_simulations=10000, min_product_errors=30, random_seed=42, model_name='ETS_Aditivo_7')

## 5. Ejecutar el pipeline integral

La función `run_pipeline` orquesta todos los módulos. No hay lógica de modelado duplicada dentro de este notebook.

In [ ]:
run_summary, output_frames = run_pipeline(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    config=CONFIG
)

display(pd.Series(run_summary, name="valor").to_frame())

,valor
fecha_ultimo_dato,2024-03-31
horizonte_inicio,2024-04-01
horizonte_fin,2024-04-07
combinaciones_sku_tienda,160
observaciones_ventas,14560
residuos_backtesting,480
combinaciones_con_pedido,156
unidades_pedido_Q90,13981
fallbacks_forecast,0


## 6. Revisar recomendación de abastecimiento Q90

La recomendación incluye una fila por SKU–Tienda, pronóstico semanal, cuantil, tope de cobertura, stock actual, inventario objetivo y pedido sugerido.

In [ ]:
recommendations = pd.read_csv(OUTPUT_DIR / "recomendacion_abastecimiento_Q90.csv")
policy_summary = pd.read_csv(OUTPUT_DIR / "resumen_politicas_abastecimiento.csv")
forecast_weekly = pd.read_csv(OUTPUT_DIR / "forecast_ets_semanal_sku_tienda.csv")

print("Combinaciones recomendadas:", len(recommendations))
print("Pedido total Q90:", int(recommendations["pedido_sugerido"].sum()))
display(policy_summary)
display(recommendations.sort_values("pedido_sugerido", ascending=False).head(20))

Combinaciones recomendadas: 160
Pedido total Q90: 13981


,politica,pedido_total_unidades,inventario_objetivo_total,combinaciones_con_pedido,costo_esperado_estimado
0,Q85,13913,17227,156,1.143526e+06
1,Q90,13981,17295,156,1.128603e+06
2,Q95,13981,17295,156,1.128603e+06


,id_tienda,id_producto,politica,nivel_cuantil,demanda_pronosticada_7d,cuantil_sin_tope,tope_cobertura,stock_actual,inventario_objetivo,pedido_sugerido,costo_esperado_estimado
0,STORE_16,PROD_005,Q90,0.9,253.999067,296.332244,291,25.0,291,266,10234.797463
1,STORE_08,PROD_007,Q90,0.9,252.603744,308.737909,289,56.0,289,233,12753.166775
2,STORE_16,PROD_006,Q90,0.9,202.717397,250.904319,232,3.0,232,229,17649.568642
3,STORE_19,PROD_005,Q90,0.9,195.586173,225.349287,224,17.0,224,207,7564.917650
4,STORE_16,PROD_007,Q90,0.9,176.734736,215.082839,202,7.0,202,195,9052.084063
5,STORE_11,PROD_005,Q90,0.9,177.318203,204.301408,203,8.0,203,195,6635.680908
6,STORE_04,PROD_006,Q90,0.9,189.445568,234.477711,217,30.0,217,187,16603.780846
7,STORE_03,PROD_007,Q90,0.9,164.607557,200.324291,189,4.0,189,185,7898.895739
8,STORE_06,PROD_007,Q90,0.9,192.277324,233.997876,220,37.0,220,183,9334.509253
9,STORE_05,PROD_008,Q90,0.9,169.481174,215.110721,194,12.0,194,182,18056.347820


## 7. Controles de salida

Se comprueba que:
- El forecast contiene una fila por combinación SKU–Tienda.
- No existen pedidos negativos.
- Los pedidos Q90 no superan el objetivo menos el stock disponible.
- Los resultados exportados están presentes.

In [ ]:
assert not forecast_weekly.duplicated(["id_tienda", "id_producto"]).any()
assert not recommendations.duplicated(["id_tienda", "id_producto"]).any()
assert (recommendations["pedido_sugerido"] >= 0).all()
assert (
    recommendations["pedido_sugerido"] ==
    (recommendations["inventario_objetivo"] - recommendations["stock_actual"]).clip(lower=0)
).all()

expected_outputs = [
    "forecast_ets_diario_7d.csv",
    "forecast_ets_semanal_sku_tienda.csv",
    "errores_semanales_backtesting_ets.csv",
    "distribucion_demanda_escenarios.csv",
    "recomendacion_abastecimiento_Q90.csv",
    "comparacion_politicas_Q85_Q90_Q95.csv",
    "resumen_politicas_abastecimiento.csv",
    "resumen_ejecucion.csv"
]
missing_outputs = [f for f in expected_outputs if not (OUTPUT_DIR / f).exists()]
assert not missing_outputs, f"Faltan salidas: {missing_outputs}"
print("Controles aprobados. Archivos generados:", len(expected_outputs))

Controles aprobados. Archivos generados: 8


## 8. Ejecución alternativa por línea de comandos

Desde la raíz del proyecto, después de instalar con `pip install -e .`:

```bash
python -m tostao_supply_chain.cli --data-dir data --output-dir outputs
```

Se pueden cambiar los parámetros principales:

```bash
python -m tostao_supply_chain.cli --data-dir data --output-dir outputs --horizon-days 7 --safety-days 1 --service-level 0.90
```

## 9. Qué falta para producción real

Este pipeline es modular y reproducible, pero aún no es un sistema productivo conectado a la operación. Antes de automatizar órdenes se requiere:
1. Integrar inventario disponible, comprometido y en tránsito.
2. Incorporar lead time y frecuencia de reposición por proveedor.
3. Incorporar múltiplos mínimos, empaques y restricciones de capacidad.
4. Modelar vida útil, merma, vencimientos y sustitución.
5. Sustituir costos proxy por costos validados con Finanzas y Abastecimiento.
6. Programar monitoreo de WAPE, sesgo, fill rate, quiebres, días de inventario y merma.

**Recomendación de implementación:** comenzar con un piloto en modo sombra: generar sugerencias, compararlas con las decisiones del equipo y medir desempeño antes de habilitar órdenes automáticas.